Sizing the 5-transistor OTA example with Evolutionary Optimization (Nevergrad) as a constraint satisfaction problem.

# Pre-body

## Clearing past runs (optional)

In [1]:
!rm -rf logs/ # clear logs
!rm -rf spice_out/

## IIC-OSIC Env Setup

In [2]:
# # The following workaround is needed to run the jupyter notebook in docker container.
import os
import subprocess

# Run the script in a shell, capture its environment
PDK = "ihp-sg13g2"
command = f"bash -c 'source /foss/tools/sak/iic-pdk-script.sh {PDK} && source ~/.bashrc && env'"
result = subprocess.run(command, capture_output=True, text=True, shell=True)

# Parse environment variables from the output
for line in result.stdout.splitlines():
    key, _, value = line.partition("=")
    if key and value:
        os.environ[key] = value
        
os.environ["PATH"] += ":/foss/tools/bin"

# Now they are in your current Python process
print("PDK_ROOT:", os.environ.get("PDK_ROOT"))
print("SPICE_USERINIT_DIR:", os.environ.get("SPICE_USERINIT_DIR"))

# Test ngspice
!ngspice -v

PDK_ROOT: /foss/pdks
SPICE_USERINIT_DIR: /foss/pdks/ihp-sg13g2/libs.tech/ngspice
******
** ngspice-44.2 : Circuit level simulation program
** Compiled with KLU Direct Linear Solver
** The U. C. Berkeley CAD Group
** Copyright 1985-1994, Regents of the University of California.
** Copyright 2001-2024, The ngspice team.
** Please get your ngspice manual from https://ngspice.sourceforge.io/docs.html
** Please file your bug-reports at http://ngspice.sourceforge.net/bugrep.html
** Creation Date: Sat May 24 09:38:33 UTC 2025
******


## Library Imports

In [3]:
import logging

from pathlib import Path

from symxplorer.spice_engine                import Spicelib_Wrapper, Sim_Execution_Type
# from symxplorer.designer_tools              import Nevergrad_Spice_Multi_Spec_Constraint_Satisfaction, Project_Setup
from symxplorer.designer_tools              import Project_Setup
from symxplorer.optimization.nevergrad      import Nevergrad_Spice_Constraint_Satisfaction
from symxplorer.logging                     import setup_loggers

logger = logging.getLogger("SymXplorer.jupyter")
logger.info("Spicelib_Wrapper imported successfully.")

# Instantiations


## Loading the project config

In [4]:
# ----------------------------
# Instantiations
# ----------------------------
project_setup_yaml = Path(f"/foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2/spice/project_setup.yaml")
_ = setup_loggers()

# (1) Load the project setup information
PROJECT_SETUP = Project_Setup.from_yaml(project_setup_yaml)
PROJECT_SETUP

05:04:33 - SymXplorer: [INFO] 🚀 Logger initialized and ready!
05:04:33 - SymXplorer: [INFO] 📄 Log file: /foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2/sizing/logs/SymXplorer_2025-10-15_05-04-33.log
05:04:33 - SymXplorer: [INFO] 🔧 spicelib logger set to 50
05:04:33 - SymXplorer.domains: [INFO] Initialized OptimizerConfig: TwoPointsDE, type=nevergrad, budget=10, random_seed=48
05:04:33 - SymXplorer.domains: [INFO] 	Linear bounds: min=0, max=100
05:04:33 - SymXplorer.domains: [INFO] 	Log bounds: min=1, max=100
05:04:33 - SymXplorer.domains: [INFO] 	Loss function: max_loss=inf, norm_method=min-max, type=mse, rescale_mag=True, include_phase_loss=False, include_mag_loss=True
05:04:33 - SymXplorer.domains: [INFO] 	Number of target specs: 3
05:04:33 - SymXplorer.domains: [INFO] 		- TargetSpec(name=ugf, target=200e6, range=1.00e+08 tolerance=10000000.0, goal=exact, sim_type=ac, enable=True, error_type=relative-sigmoid, weight=100.0, enable=True, description=Unitiy gain frequency)
05:04

Project_Setup(name='5T-OTA', description='5 Transistor OTA example sizing in the ihp-sg13g2 technology', simulator='/foss/tools/bin/ngspice', ws_root=PosixPath('/foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2'), netlist=PosixPath('spice/ota-5t_tb-loopgain.spice'), outdir=PosixPath('sizing/spice_out'), tech_spec=TechSpec(name='ihp-sg13g2', constraints={'max_nfet_w': np.float64(9.999999999999999e-06), 'min_nfet_w': np.float64(1.8e-07), 'max_nfet_l': np.float64(9.999999999999999e-06), 'min_nfet_l': np.float64(1.8e-07), 'max_pfet_w': np.float64(9.999999999999999e-06), 'min_pfet_w': np.float64(1.8e-07), 'max_pfet_l': np.float64(9.999999999999999e-06), 'min_pfet_l': np.float64(1.8e-07)}), pvt=PVT(temp=25, corner='tt', supply=1.8), dut_params=[Param(name='x_dut_nfet_input_w', min_val=np.float64(1.8e-07), max_val=np.float64(9.999999999999999e-06), val=None, description=None, log_scale=False), Param(name='x_dut_nfet_input_l', min_val=np.float64(1.8e-07), max_val=np.float64(9.99999999999

## Create a SPICE simulator wrapper

In [5]:
# (2) Create the Spice Simulator Wrapper
netlist_filename = Path(PROJECT_SETUP.ws_root) / Path(PROJECT_SETUP.netlist)
output_folder    = Path(PROJECT_SETUP.ws_root) / Path(PROJECT_SETUP.outdir)

wrapper = Spicelib_Wrapper(
    project_name=PROJECT_SETUP.name,
    netlist_filename=netlist_filename,
    output_folder=output_folder,
    sim_execution_t=Sim_Execution_Type.RUN_AND_WAIT,  # only RUN_AND_WAIT is supported as of now...,
    path_to_simulator=Path("/foss/tools/bin/ngspice"),
    verbose=False
    )
wrapper

05:04:33 - SymXplorer.spicelib: [INFO] 📂 Creating output directory for the first time: /foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2/sizing/spice_out
05:04:33 - SymXplorer.spicelib: [INFO] --------------------------------------------------
05:04:33 - SymXplorer.spicelib: [INFO] 🚀 Spicelib_Wrapper initialized successfully!
05:04:33 - SymXplorer.spicelib: [INFO] 	📝 Project: 5T-OTA
05:04:33 - SymXplorer.spicelib: [INFO] 	📜 Schematic: ota-5t_tb-loopgain
05:04:33 - SymXplorer.spicelib: [INFO] 	📂 Output Folder: /foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2/sizing/spice_out
05:04:33 - SymXplorer.spicelib: [INFO] --------------------------------------------------
05:04:34 - SymXplorer.spicelib: [INFO] Using ngspice from ['/foss/tools/bin/ngspice']
05:04:34 - SymXplorer.spicelib: [INFO] 📊 --- Circuit Information ---
05:04:34 - SymXplorer.spicelib: [INFO] 🔗 Nodes in the netlist: ['v_dd', 'GND', 'v_ss', 'v_in', 'v_ena', 'vr1', 'net1', 'vf1', 'net2', 'net3', 'net4', 'net5', 'ne

## Create an optimizer object

In [6]:
circuit_optimizer = Nevergrad_Spice_Constraint_Satisfaction(
    spicelib_wrapper=wrapper,
    setup_obj=PROJECT_SETUP
)
circuit_optimizer

05:04:34 - SymXplorer.base_optimizer: [INFO] Initialized the Nevergrad_Spice_Multi_Spec_Optimizer with 3 target specs
05:04:34 - SymXplorer.Nevergrad: [INFO] started the <class 'symxplorer.optimization.nevergrad.Nevergrad_Spice_Constraint_Satisfaction'> optimizer class


## Sanity Check

In [7]:
wrapper.run_sanity_check(
    use_editor=True,
    sim_execution_t=Sim_Execution_Type.RUN_NOW
)

05:04:34 - SymXplorer.spicelib: [INFO] 📂 Creating dedicated sanity check folder...
05:04:34 - SymXplorer.spicelib: [INFO] 🧪 Running sanity check simulation...
05:04:34 - SymXplorer.spicelib: [INFO] simulator log: /foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2/sizing/spice_out/sanity_check/5T-OTA_sanity.log
05:04:34 - SymXplorer.spicelib: [INFO] simulator RAW: /foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2/sizing/spice_out/sanity_check/5T-OTA_sanity.raw
05:04:34 - SymXplorer.spicelib: [INFO] 🔎 Verifying simulation results...
05:04:34 - SymXplorer.spicelib: [INFO] ✅ Sanity check passed 🎉


True

# Main Body

## Optimization

In [8]:
circuit_optimizer.parameterize()

Dict(x_dut_nfet_input_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_nfet_input_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_nfet_mirror_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_nfet_mirror_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_pfet_load_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_pfet_load_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}]):{'x_dut_nfet_input_w': 50.0, 'x_dut_nfet_input_l': 50.0, 'x_dut_nfet_mirror_w': 50.0, 'x_dut_nfet_mirror_l': 50.0, 'x_dut_pfet_load_w': 50.0, 'x_dut_pfet_load_l': 50.0}

In [9]:
_ = circuit_optimizer.optimize()

05:04:34 - SymXplorer.base_optimizer: [INFO] Optimization process started.
05:04:34 - SymXplorer.Nevergrad: [INFO] Optimizer is set to TwoPointsDE with budget = 10
Optimizing: 100%|██████████| 10/10 [00:07<00:00,  1.38trial/s]
05:04:42 - SymXplorer.base_optimizer: [INFO] Optimization process completed.


In [10]:
circuit_optimizer.plot_score(save_path=project_setup_yaml.parent / "loss_curve.html", show=True)

05:04:43 - SymXplorer.plotter: [INFO] 📊 Plot saved to /foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2/spice/loss_curve.html
05:04:43 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


## Inspection & Visualization

### (1) Best Param

In [11]:
out = circuit_optimizer.get_best_params()
if out is not None: 
    best_param, loss, metadata = out
# metadata

05:04:43 - SymXplorer.base_optimizer: [INFO] best score: -52.07452993067651


In [12]:
# Print parameter sizes (convert to u)
for param in best_param:
    print(f"{param}: {best_param[param] :0.2e}")

x_dut_nfet_input_w: 1.80e-07
x_dut_nfet_input_l: 1.80e-07
x_dut_nfet_mirror_w: 1.80e-07
x_dut_nfet_mirror_l: 1.80e-07
x_dut_pfet_load_w: 1.80e-07
x_dut_pfet_load_l: 1.80e-07


In [13]:
circuit_optimizer.plot_solution(best_param, show_plot=True, trace_name="out")

05:04:44 - SymXplorer.base_optimizer: [INFO] total score: -34.25130648189347
05:04:44 - SymXplorer.base_optimizer: [INFO] 	Spec 'ugf': curr_val=126434000.0, score=-30.754347972766325
05:04:44 - SymXplorer.base_optimizer: [INFO] 	Spec 'dcgain': curr_val=23.003230000000002, score=-3.496958509127146
05:04:44 - SymXplorer.base_optimizer: [INFO] 	Spec 'pm': curr_val=87.0, score=0.0


### (3) Metric Trace

In [14]:
_ = circuit_optimizer.plot_optimization_trace(metric_x='pm', metric_y='ugf', show=True)

05:04:44 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


In [15]:
circuit_optimizer.plot_score_value_by_spec(spec_name="dcgain", show=True)
circuit_optimizer.plot_score_value_by_spec(spec_name="ugf", show=True)
circuit_optimizer.plot_score_value_by_spec(spec_name="pm", show=True)

05:04:45 - SymXplorer.plotter: [INFO] 	min score -1.5068859280010782; max score 0.0
05:04:45 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


05:04:45 - SymXplorer.plotter: [INFO] 	min score -70.9326806031895; max score -48.34578750610794
05:04:45 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


05:04:45 - SymXplorer.plotter: [INFO] 	min score -20.80297712989976; max score 0.0
05:04:45 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


### (4) Design Space Exploration

In [16]:
circuit_optimizer.plot_design_space_exploration(param_x="x_dut_pfet_load_w", param_y="x_dut_pfet_load_l", show=True)
circuit_optimizer.plot_design_space_exploration(param_x="x_dut_nfet_input_w", param_y="x_dut_nfet_input_l", show=True)
circuit_optimizer.plot_design_space_exploration(param_x="x_dut_nfet_mirror_w", param_y="x_dut_nfet_mirror_l", show=True)

05:04:45 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


05:04:46 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


05:04:46 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


(tensor([1.8000e-07, 1.8000e-07, 1.8000e-07, 1.8000e-07, 1.8000e-07, 1.8000e-07,
         1.8000e-07, 1.8000e-07, 1.8000e-07, 1.8000e-07]),
 tensor([1.8000e-07, 1.8000e-07, 1.8000e-07, 1.8000e-07, 1.8000e-07, 1.8000e-07,
         1.8000e-07, 1.8000e-07, 1.8000e-07, 1.8000e-07]))

# Checkpointing

In [17]:
name = str(PROJECT_SETUP.ws_root/Path("sizing/data/checkpoint"))
circuit_optimizer.save_checkpoint(name=name)

05:04:46 - SymXplorer.base_optimizer: [INFO] ✅ Checkpoint saved to /foss/designs/eda/SymXplorer/examples/5t-ota/ihp-sg13g2/sizing/data/checkpoint_2025-10-15_05-04-46.json
